# MicroGPT — 순수 Python으로 구현하는 GPT

> 원본: [Andrej Karpathy's MicroGPT Gist](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)

## 개요

이 노트북은 **외부 딥러닝 라이브러리 없이 순수 Python만으로** GPT(Generative Pre-trained Transformer)를 구현합니다.

### 왜 이것이 의미 있는가?

| 구성 요소 | 일반적 구현 (PyTorch 등) | 이 노트북 |
|---|---|---|
| 자동 미분 (Autograd) | `torch.autograd` | `Value` 클래스 직접 구현 |
| 신경망 레이어 | `torch.nn.Linear`, `torch.nn.LayerNorm` | 리스트 연산으로 직접 구현 |
| Transformer 구조 | `torch.nn.TransformerEncoder` | `gpt()` 함수로 직접 구현 |
| 최적화 알고리즘 | `torch.optim.Adam` | Adam 직접 구현 |

### 학습 목표
1. **Autograd (자동 미분)**: 계산 그래프와 역전파(Backpropagation)의 원리
2. **Transformer 아키텍처**: Self-Attention, Multi-Head Attention, MLP 블록
3. **GPT 모델**: 언어 생성 모델의 구조와 학습 과정
4. **Adam 최적화**: 적응적 학습률 최적화 알고리즘

## 1단계: 데이터셋과 토큰화 (Tokenization)

### 데이터셋
- **이름 데이터셋**: Andrej Karpathy의 `makemore` 프로젝트에서 가져온 약 32,000개의 영어 이름
- 각 이름은 하나의 "문서(document)"로 취급됩니다

### 토큰화 (Tokenization)란?
언어 모델은 텍스트를 직접 처리할 수 없으므로 텍스트를 **정수(token ID)**로 변환합니다.

```
"anna" → [0, 13, 13, 0]  (각 문자 → 정수 ID)
```

### 이 구현의 토큰화 방식

| 개념 | 설명 |
|---|---|
| **Character-level** | 각 고유 문자가 하나의 토큰 (a=0, b=1, ..., z=25) |
| **BOS 토큰** | **B**eginning **O**f **S**equence — 시퀀스의 시작과 끝을 표시하는 특수 토큰 (ID=26) |
| **vocab_size** | 고유 문자 수 + 1 (BOS) = 27 |

> **참고**: 실제 GPT 모델은 **BPE (Byte Pair Encoding)** 토큰화를 사용하여 50,000개 이상의 토큰을 가집니다. 여기서는 단순화를 위해 문자 단위 토큰화를 사용합니다.

In [ ]:
import os       # os.path.exists
import math     # math.log, math.exp
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos

# Let there be a Dataset `docs`: list[str] of documents (e.g. a list of names)
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

# Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
vocab_size = len(uchars) + 1 # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

## 2단계: Autograd — 자동 미분 엔진

### 역전파 (Backpropagation)와 계산 그래프

신경망 학습의 핵심은 **손실 함수(Loss)를 최소화**하는 것입니다. 이를 위해 각 파라미터에 대한 **기울기(gradient)**를 계산해야 합니다.

### 연쇄 법칙 (Chain Rule)

$f(g(x))$의 미분은:

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

이것이 **역전파의 수학적 기초**입니다. 복잡한 함수도 간단한 연산의 체인으로 분해하면, 각 연산의 **로컬 미분(local gradient)**만 알면 됩니다.

### `Value` 클래스의 구조

| 속성 | 역할 |
|---|---|
| `data` | 순전파(forward pass)에서 계산된 **스칼라 값** |
| `grad` | 역전파에서 계산된 **기울기** ($\frac{\partial \text{Loss}}{\partial \text{this node}}$) |
| `_children` | 계산 그래프에서 이 노드의 **자식 노드들** |
| `_local_grads` | 이 노드의 자식에 대한 **로컬 미분 값들** |

### 각 연산의 미분 규칙

| 연산 | 순전파 | 역전파 (로컬 미분) |
|---|---|---|
| $c = a + b$ | `a.data + b.data` | $\frac{\partial c}{\partial a} = 1, \quad \frac{\partial c}{\partial b} = 1$ |
| $c = a \times b$ | `a.data * b.data` | $\frac{\partial c}{\partial a} = b, \quad \frac{\partial c}{\partial b} = a$ |
| $c = a^n$ | `a.data ** n` | $\frac{\partial c}{\partial a} = n \cdot a^{n-1}$ |
| $c = \log(a)$ | `math.log(a.data)` | $\frac{\partial c}{\partial a} = \frac{1}{a}$ |
| $c = e^a$ | `math.exp(a.data)` | $\frac{\partial c}{\partial a} = e^a$ |
| $c = \text{ReLU}(a)$ | `max(0, a.data)` | $\frac{\partial c}{\partial a} = \begin{cases} 1 & \text{if } a > 0 \\ 0 & \text{otherwise} \end{cases}$ |

### `backward()` 메서드 — 위상 정렬(Topological Sort)

역전파는 계산 그래프를 **역순(reverse topological order)**으로 순회하면서 기울기를 전파합니다:

1. **위상 정렬**: DFS로 계산 그래프의 노드를 정렬
2. **기울기 초기화**: 손실 노드의 기울기를 1로 설정 ($\frac{\partial L}{\partial L} = 1$)
3. **역순 순회**: 각 노드에서 자식으로 기울기를 전파 (`child.grad += local_grad * v.grad`)

> **핵심**: `+=`를 사용하는 이유는 하나의 노드가 여러 경로에서 사용될 수 있기 때문입니다 (기울기 누적).

In [ ]:
# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

## 3단계: 모델 파라미터 초기화

### Transformer의 하이퍼파라미터

| 하이퍼파라미터 | 값 | 설명 |
|---|---|---|
| `n_layer` | 1 | Transformer 블록의 수 (깊이) |
| `n_embd` | 16 | 임베딩 차원 (폭) |
| `block_size` | 16 | 최대 컨텍스트 길이 (attention이 볼 수 있는 최대 토큰 수) |
| `n_head` | 4 | Multi-Head Attention의 헤드 수 |
| `head_dim` | 4 | 각 헤드의 차원 = `n_embd / n_head` |

### 파라미터 행렬들 (가중치)

GPT 모델은 아래와 같은 학습 가능한 파라미터(가중치 행렬)들로 구성됩니다:

| 파라미터 | 크기 | 역할 |
|---|---|---|
| `wte` | (27, 16) | **Token Embedding** — 각 토큰을 벡터로 변환 |
| `wpe` | (16, 16) | **Position Embedding** — 위치 정보를 벡터로 인코딩 |
| `attn_wq` | (16, 16) | Attention **Query** 행렬 |
| `attn_wk` | (16, 16) | Attention **Key** 행렬 |
| `attn_wv` | (16, 16) | Attention **Value** 행렬 |
| `attn_wo` | (16, 16) | Attention **Output** 투영 행렬 |
| `mlp_fc1` | (64, 16) | MLP 첫 번째 레이어 (확장: 16→64) |
| `mlp_fc2` | (16, 64) | MLP 두 번째 레이어 (축소: 64→16) |
| `lm_head` | (27, 16) | **Language Model Head** — 임베딩을 토큰 확률로 변환 |

> **참고**: 모든 가중치는 평균 0, 표준편차 0.08의 가우시안 분포로 랜덤 초기화됩니다. 각 값은 `Value` 객체로 감싸져 자동 미분이 가능합니다.

In [ ]:
# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # width of the network (embedding dimension)
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
print(f"num params: {len(params)}")

## 4단계: GPT 모델 아키텍처

### Transformer 아키텍처 개요

GPT는 **Transformer Decoder** 구조를 기반으로 합니다. 각 토큰은 다음 과정을 거칩니다:

```
입력 토큰 → [Token Embedding + Position Embedding] → RMSNorm → [Attention Block → MLP Block] × N → Linear Head → 다음 토큰 확률
```

### 4.1 임베딩 (Embedding)

```python
tok_emb = wte[token_id]   # 토큰 → 16차원 벡터
pos_emb = wpe[pos_id]     # 위치 → 16차원 벡터
x = tok_emb + pos_emb     # 합산하여 위치 정보 포함
```

- **Token Embedding**: "이 토큰이 무엇인가?"
- **Position Embedding**: "이 토큰이 어디에 있는가?"

### 4.2 RMSNorm (Root Mean Square Normalization)

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}$$

- 벡터의 크기(magnitude)를 정규화하여 학습을 안정화합니다
- GPT-2의 LayerNorm을 단순화한 변형입니다

### 4.3 Self-Attention (자기 어텐션) ⭐

Attention은 Transformer의 **핵심 메커니즘**입니다.

**직관적 설명**: "이 토큰이 다른 토큰들 중 어디에 주목해야 하는가?"

각 토큰에서 3개의 벡터를 생성합니다:
- **Query (Q)**: "나는 무엇을 찾고 있는가?" 
- **Key (K)**: "나는 무엇을 제공하는가?"
- **Value (V)**: "내가 실제로 전달할 정보"

**Attention 계산**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

1. Q와 K의 내적 → 유사도 점수 (attention logits)
2. $\sqrt{d_k}$로 나누기 → 스케일링 (값이 너무 커지는 것을 방지)
3. Softmax → 확률 분포로 변환 (attention weights)
4. V에 가중합 → 최종 출력

### 4.4 Multi-Head Attention

하나의 큰 어텐션 대신 **여러 개의 작은 어텐션 "헤드"**를 병렬로 실행합니다:

```
n_head = 4, head_dim = 4 (= 16 / 4)

Head 1: Q[0:4], K[0:4], V[0:4]   → 문법적 관계에 주목
Head 2: Q[4:8], K[4:8], V[4:8]   → 의미적 관계에 주목  
Head 3: Q[8:12], K[8:12], V[8:12] → 다른 패턴에 주목
Head 4: Q[12:16], K[12:16], V[12:16] → 또 다른 패턴에 주목
```

각 헤드가 서로 다른 "관점"에서 정보를 수집합니다.

### 4.5 MLP 블록 (Feed-Forward Network)

```
x → Linear(16→64) → ReLU → Linear(64→16) → x
```

- Attention이 "어디를 볼지" 결정한 후, MLP가 "무엇을 할지" 처리합니다
- 4배로 확장 후 다시 축소하는 **bottleneck** 구조
- **ReLU**: $\text{ReLU}(x) = \max(0, x)$ (비선형 활성화 함수)

### 4.6 Residual Connection (잔차 연결)

```python
x = attention_output + x_residual  # 잔차 연결
```

- 입력을 출력에 **더해서** 정보가 직접 흘러가는 "지름길"을 만듭니다
- 깊은 네트워크에서 **기울기 소실(vanishing gradient)** 문제를 완화합니다

### Softmax 함수

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

- 임의의 실수 벡터를 **확률 분포**로 변환 (모든 값이 0~1이고 합이 1)
- `max(x)` 를 빼는 것은 수치 안정성을 위한 트릭입니다

In [ ]:
# Define the model architecture: a function mapping tokens and parameters to logits over what comes next
# Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, no biases, GeLU -> ReLU
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

## 5단계: 학습 (Training) 및 추론 (Inference)

### 학습 루프 개요

```
for each step:
    1. 문서 선택 → 토큰화
    2. Forward Pass: 토큰 → GPT → logits → softmax → 확률 → 손실(loss)
    3. Backward Pass: loss.backward() → 모든 파라미터의 기울기 계산
    4. Adam 최적화: 기울기를 이용해 파라미터 업데이트
```

### 5.1 손실 함수 (Cross-Entropy Loss)

모델은 각 위치에서 **다음 토큰을 예측**합니다:

$$L = -\frac{1}{n}\sum_{t=1}^{n} \log P(\text{target}_t)$$

- $P(\text{target}_t)$: 모델이 정답 토큰에 부여한 확률
- 정답 확률이 높을수록 → $\log$ 값이 0에 가까움 → 손실이 낮음
- **목표**: 이 손실을 최소화하는 것

### 5.2 Adam 최적화 알고리즘

Adam은 **적응적 학습률**을 사용하는 최적화 알고리즘입니다:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(1차 모멘트, 기울기의 이동 평균)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(2차 모멘트, 기울기 제곱의 이동 평균)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(편향 보정)}$$
$$\theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} \quad \text{(파라미터 업데이트)}$$

| 기호 | 값 | 의미 |
|---|---|---|
| $\alpha$ | 0.01 | 학습률 (선형 감소 적용) |
| $\beta_1$ | 0.85 | 1차 모멘트 감쇠율 |
| $\beta_2$ | 0.99 | 2차 모멘트 감쇠율 |
| $\epsilon$ | 1e-8 | 0으로 나누기 방지 |

> **직관**: SGD는 모든 파라미터에 같은 학습률을 적용하지만, Adam은 각 파라미터마다 **과거 기울기 이력**을 고려해 학습률을 자동 조절합니다.

### 5.3 추론 (Inference) — 텍스트 생성

학습된 모델로 새로운 이름을 생성합니다:

1. BOS 토큰으로 시작
2. 모델이 다음 토큰의 확률 분포를 출력
3. **Temperature**를 적용하여 확률 분포를 조절:
   - `temperature → 0`: 가장 높은 확률의 토큰만 선택 (결정적)
   - `temperature = 1`: 원래 확률 분포대로 샘플링
   - `temperature > 1`: 더 균등한 분포 (더 "창의적")
4. 확률 분포에서 랜덤 샘플링
5. BOS 토큰이 나올 때까지 반복

$$P'(x_i) = \text{softmax}\left(\frac{\text{logit}_i}{T}\right) \quad \text{where } T = \text{temperature}$$

In [ ]:
# Let there be Adam, the blessed optimizer and its buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # first moment buffer
v = [0.0] * len(params) # second moment buffer

# Repeat in sequence
num_steps = 1000 # number of training steps
for step in range(num_steps):

    # Take single document, tokenize it, surround it with BOS special token on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update: update the model parameters based on the corresponding gradients
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')

# Inference: may the model babble back to us
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")